# Chart2Data Fine-tuning with Unsloth

This notebook fine-tunes a vision language model on the Chart2Data dataset to
extract numerical time-series data from chart images.

**Pipeline:** Load dataset → Format as chat messages → Fine-tune with LoRA →
Evaluate on validation split → Push adapter and merged model to HuggingFace →
Inference with Unsloth Studio UI.

## Configuration
Change the parameters below before running. All models supported by Unsloth work.

| Parameter | Recommended values |
|---|---|
| MODEL_ID | `unsloth/Qwen3.5-0.8B` (free), `unsloth/Qwen3.5-2B`, `unsloth/Qwen3.5-4B` |
| MAX_SEQ_LEN | 1024 (0.8B), 2048 (2B), 4096 (4B) |
| EPOCHS | 2-5 for small dataset, 3-8 for larger |
| BATCH_SIZE | 1-4 (adjust based on available VRAM) |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Configuration — change these to match your hardware and goals
# ═══════════════════════════════════════════════════════════════════════════

# Model to fine-tune
MODEL_ID = "unsloth/Qwen3.5-0.8B-Instruct"  # Change to "unsloth/Qwen3.5-2B-Instruct" or "unsloth/Qwen3.5-4B-Instruct"

# Dataset on HuggingFace
DATASET_REPO = "maurovidal/chart2data-v0-sft"  # Change to "maurovidal/chart2data-v0.2-sft" after publishing v0.2
DATASET_SPLIT_TRAIN = "train"
DATASET_SPLIT_VALIDATION = "validation"

# Output model repository on HuggingFace
MODEL_REPO = "maurovidal/chart2data-qwen3.5-0.8b"  # Change per model size
MODEL_REVISION = "v0.1"  # Tag for this training run
MODEL_USERNAME = "maurovidal"  # Your HuggingFace username (for pushing adapters)

# Training hyperparameters
MAX_SEQ_LEN = 1024  # 1024 for 0.8B, 2048 for 2B, 4096 for 4B
EPOCHS = 3
BATCH_SIZE = 2  # Adjust based on GPU memory; use gradient_accumulation_steps=4 to compensate
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = BATCH_SIZE * ACCUMULATION_STEPS
LEARNING_RATE = 2e-4  # Good starting point for LoRA fine-tuning
LORA_R = 32  # LoRA rank (higher = more parameters, potentially better quality)
LORA_ALPHA = 64  # LoRA alpha (scales LoRA weights, typically 2x r)
LORA_DROPOUT = 0.05  # Regularization for LoRA layers
WEIGHT_DECAY = 0.01  # L2 regularization on non-LoRA parameters
WARMUP_RATIO = 0.05  # Linear warmup fraction of total steps

# Evaluation settings
EVAL_STEPS = 50  # Evaluate every N steps
SAVE_STEPS = 50  # Save checkpoint every N steps
MAX_EVAL_SAMPLES = 200  # Number of validation samples to evaluate (use fewer for faster eval)

print("Configuration loaded.")
print(f"  Model: {MODEL_ID}")
print(f"  Dataset: {DATASET_REPO}")
print(f"  Output repo: {MODEL_REPO} (revision: {MODEL_REVISION})")
print(f"  Epochs: {EPOCHS} | Batch: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS}")
print(f"  LoRA: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"  LR: {LEARNING_RATE} | Seq len: {MAX_SEQ_LEN}")

## Step 1: Install dependencies

In [ ]:
!pip install -q unsloth trl datasets peft transformers accelerate bitsandbytes

## Step 2: Import libraries

In [ ]:
import json
import numpy as np
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

## Step 3: HuggingFace login
Enter your HuggingFace token with write permission. Get one at: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login

# If you have a token saved from previous use, it may auto-login.
# If not, paste your token in the popup that appears.
try:
    login()
    print("✓ HuggingFace logged in")
except Exception as e:
    print(f"Login needed: {e}")
    token = input("Paste your HF token: ")
    login(token=token)

## Step 4: Load and format the dataset

In [ ]:
print(f"Loading dataset: {DATASET_REPO}")
dataset = load_dataset(DATASET_REPO, split="train")
print(f"Train set: {len(dataset):,} samples")
print(f"Columns: {dataset.column_names}")
print()
print("Sample row:")
print(dataset[0].to_dict())

## Step 5: Load model and tokenizer

In [ ]:
# Load model with 4-bit quantization to save memory
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,  # Auto-detect; use dtype="float16" for A100/V100
    load_in_4bit=True,  # 4-bit quantization to fit in Colab free GPU
)

print(f"Model loaded: {MODEL_ID}")
print(f"  Dtype: {model.dtype}")
print(f"  Max sequence length: {MAX_SEQ_LEN}")
print(f"  Vocabulary size: {len(tokenizer)}")

## Step 6: Apply LoRA adapters

In [ ]:
# Apply LoRA adapters with the configured hyperparameters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory-efficient gradient checkpointing
    use_rslora=False,  # Set True for rank-stabilized LoRA
    loftq_config=None,  # Set to LoftQ config for higher precision 4-bit if available
)

# Print trainable parameters for verification
model.print_trainable_parameters()

## Step 7: Configure and run training

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="messages",  # The column containing chat messages
    max_seq_length=MAX_SEQ_LEN,
    data_collator="smart",  # Smart padding and batching
    dataset_num_proc=2,  # Number of parallel data processing workers
    max_steps=0,  # 0 = run for exactly n_epochs
    packing=False,  # Don't pack sequences (important for image datasets)
    args=SFTConfig(
        output_dir="./checkpoints",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_steps=EVAL_STEPS,
        evaluation_strategy="steps",
        save_steps=SAVE_STEPS,
        load_best_model_at_end=True,
        report_to="none",  # Set to "tensorboard" or "wandb" for tracking
        fp16=True,  # Mixed precision training
        bf16=False,
        metric_for_best_model="loss",  # Save best model by validation loss
        save_strategy="steps",
    ),
)

In [ ]:
print("Starting training...")
print("This may take 30-90 minutes depending on model size and dataset size.")
print()
trainer.train()

## Step 8: Evaluate on validation set
Check how well the fine-tuned model performs on unseen chart images.

### Evaluation metrics definition

In [ ]:
def eval_json_parse(text: str) -> dict:
    """Try to parse a JSON response from the model."""
    try:
        # Handle code blocks
        text = text.strip()
        if text.startswith("```"):
            lines = text.split("\n")
            for i, line in enumerate(lines):
                if line.strip().startswith("json"):
                    text = "\n".join(lines[i+1:])
                    break
            text = text.split("```")[0]
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def mae(actual: list[float], predicted: list[float]) -> float:
    """Mean Absolute Error between two arrays."""
    if len(actual) != len(predicted):
        return float("inf")
    return float(np.mean(np.abs(np.array(actual) - np.array(predicted))))

def rmse(actual: list[float], predicted: list[float]) -> float:
    """Root Mean Squared Error between two arrays."""
    if len(actual) != len(predicted):
        return float("inf")
    return float(np.sqrt(np.mean((np.array(actual) - np.array(predicted))**2)))

def normalized_mae(actual: list[float], predicted: list[float]) -> float:
    """MAE normalized by the value range."""
    if len(actual) != len(predicted):
        return float("inf")
    span = max(actual) - min(actual) if max(actual) != min(actual) else 1.0
    return float(np.mean(np.abs(np.array(actual) - np.array(predicted))) / span)

def parse_success_rate(samples: list[dict]) -> float:
    """Fraction of model responses that are valid JSON."""
    if not samples:
        return 0.0
    parsed = [eval_json_parse(s["assistant"]) for s in samples]
    return sum(1 for p in parsed if p is not None) / len(samples)

### Run inference on a validation subset

In [ ]:
from datasets import load_dataset

# Load validation set
validation_dataset = load_dataset(DATASET_REPO, split=DATASET_SPLIT_VALIDATION)
print(f"Validation set: {len(validation_dataset):,} samples")

# Sample a subset for faster evaluation
eval_samples = validation_dataset.select(range(min(MAX_EVAL_SAMPLES, len(validation_dataset))))
print(f"Evaluating on {len(eval_samples)} samples...")
print()

In [ ]:
import torch
import PIL.Image
import base64
from io import BytesIO

# Apply prompt template for inference
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    mapping="mroon",  # Use mroon mapping for Qwen3.5
    map_location=model.device,  # Use GPU for faster generation
)

# Enable inference mode (disable gradient computation)
FastLanguageModel.for_inference(model)

In [ ]:
# Collect predictions
predictions = []
correct = 0
total = 0
mae_values = []
rmse_values = []
nmae_values = []
parse_count = 0

print("Running inference on evaluation samples...")
for i, row in enumerate(eval_samples):
    # Get the instruction (user message)
    user_content = row["messages"][0]["content"]
    ground_truth = row["messages"][1]["content"][0]["text"]  # assistant message
    
    # Build conversation for the model
    messages = [
        {"role": "user", "content": user_content},
    ]
    
    # Apply the chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    # Generate response
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.1,
        top_p=0.9,
        do_sample=True,
    )
    
    # Decode the generated response
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract the model's JSON output
    # The model should output JSON after the chat template
    if "assistant" in generated:
        json_part = generated.split("assistant")[-1].strip()
        if json_part.startswith(""):
            json_part = json_part.split("")[-1]
        json_part = json_part.split("\n")[0]  # First line
    else:
        json_part = generated  # Use full generation if no assistant tag
    
    # Try to parse
    predicted_data = eval_json_parse(json_part)
    if predicted_data is not None and "x" in predicted_data and "y" in predicted_data:
        try:
            ground_truth_data = json.loads(ground_truth)
            if "x" in ground_truth_data and "y" in ground_truth_data:
                # Compare x and y arrays
                if len(predicted_data["x"]) == len(ground_truth_data["x"]) and len(predicted_data["y"]) == len(ground_truth_data["y"]):
                    mae_x = mae(ground_truth_data["x"], predicted_data["x"])
                    mae_y = mae(ground_truth_data["y"], predicted_data["y"])
                    mae_values.append((mae_x + mae_y) / 2)
                    rmse_values.append(rmse(ground_truth_data["y"], predicted_data["y"]))
                    nmae_values.append(normalized_mae(ground_truth_data["y"], predicted_data["y"]))
                    correct += 1
                    parse_count += 1
        except Exception:
            pass
    total += 1
    
    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(eval_samples)} samples")

# Compute and display metrics
eval_metrics = {
    "total_samples": total,
    "successful_parses": parse_count,
    "parse_success_rate": parse_success_rate([{"assistant": "[parsed]"} for _ in range(parse_count)]),
    "exact_match_rate": correct / total if total > 0 else 0,
    "mae_x": np.mean(mae_values) if mae_values else 0,
    "mae_y": np.mean(mae_values) if mae_values else 0,
    "rmse_y": np.mean(rmse_values) if rmse_values else 0,
    "nmae_y": np.mean(nmae_values) if nmae_values else 0,
}

print("\n" + "="*60)
print("Evaluation Results")
print("="*60)
for key, value in eval_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")
print("="*60)

## Step 9: Save and push to HuggingFace

### Save the LoRA adapter

In [ ]:
# Save the fine-tuned adapter locally
adapter_path = f"./lora_{MODEL_ID.split('/')[-1]}_{MODEL_REVISION}"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"✓ Adapter saved locally to: {adapter_path}")
print(f"  Path size: {sum(p.numel() for p in model.parameters() if p.requires_grad) * 2 / 1024 / 1024:.2f} MB")

### Push adapter to HuggingFace

In [ ]:
# Create model repository on HuggingFace (will be created if it doesn't exist)
from huggingface_hub import create_repo, upload_folder
import shutil

try:
    create_repo(
        repo_id=MODEL_REPO,
        exist_ok=True,
        repo_type="model",
    )
    print(f"✓ Model repo ready: {MODEL_REPO}")
except Exception as e:
    print(f"Repo may already exist or error: {e}")

# Push the adapter
upload_folder(
    folder_path=adapter_path,
    repo_id=f"{MODEL_USERNAME}/{MODEL_REPO.split('/')[-1]}",
    revision=MODEL_REVISION,
    create_pr=False,
)
print(f"✓ Adapter uploaded to: https://huggingface.co/{MODEL_USERNAME}/{MODEL_REPO.split('/')[-1]}/tree/{MODEL_REVISION}")

### Merge and push full 16-bit model (optional but recommended)
This combines the base model and LoRA adapter into a single ready-to-use model.

In [ ]:
# Uncomment and run the following to merge and push full model:
# print("Merging adapter with base model (this may take a few minutes)...")
# model.save_pretrained_merged(f"{adapter_path}/merged", tokenizer, save_method="merged_16bit")
# upload_folder(
#     folder_path=f"{adapter_path}/merged",
#     repo_id=f"{MODEL_USERNAME}/{MODEL_REPO.split('/')[-1]}",
#     revision=f"{MODEL_REVISION}-merged",
#     create_pr=False,
# )
# print(f"✓ Merged model uploaded to: https://huggingface.co/{MODEL_USERNAME}/{MODEL_REPO.split('/')[-1]}/tree/{MODEL_REVISION}-merged")

## Step 10: Test with Unsloth Studio (Chat UI)

In [ ]:
# Launch interactive chat with the fine-tuned model
# Use Ctrl+D to exit the chat

from unsloth.chat_templates import get_chat_template
from IPython.display import display, Markdown

# Apply chat template
tokenizer = get_chat_template(
    tokenizer,
    mapping="mroon",
)

# Enable inference mode
FastLanguageModel.for_inference(model)

print("="*60)
print("Chat with your fine-tuned model")
print("Type your chart image path or question, and 'exit' to quit.")
print("="*60)
print()

conversation_history = []
while True:
    user_input = input("\nYou: ").strip()
    if user_input.lower() in ["exit", "quit"]:
        print("\nGoodbye! 👋")
        break
    
    # Add to conversation history
    conversation_history.append({"role": "user", "content": user_input})
    
    # Apply chat template
    text = tokenizer.apply_chat_template(
        conversation_history,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    # Generate response
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
    )
    
    # Decode and display
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant's response
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    
    print(f"\nModel: {response}")
    conversation_history.append({"role": "assistant", "content": response})
    print()

print("\nChat complete! Your fine-tuned model is saved and ready to use.")
print(f"Adapter: {adapter_path}")
print(f"Hub URL: https://huggingface.co/{MODEL_USERNAME}/{MODEL_REPO.split('/')[-1]}/tree/{MODEL_REVISION}")

## Summary

You've just:
1. ✅ Loaded the Chart2Data dataset
2. ✅ Applied LoRA fine-tuning to a vision language model
3. ✅ Evaluated model performance on validation data
4. ✅ Saved the adapter and pushed to HuggingFace
5. ✅ Tested the model in interactive chat mode

### Next steps
- **Try different model sizes:** Change `MODEL_ID` to `unsloth/Qwen3.5-2B-Instruct` or `unsloth/Qwen3.5-4B-Instruct` and re-run.
- **Adjust hyperparameters:** Increase `EPOCHS` for more training, or adjust `LEARNING_RATE` and `LORA_R`.
- **Publish your model:** Visit https://huggingface.co/{MODEL_USERNAME}/{MODEL_REPO.split('/')[-1]} to view and share your fine-tuned model.
- **Use with Unsloth Studio:** Load the adapter with `FastLanguageModel.from_pretrained(MODEL_REPO, revision=MODEL_REVISION)` for fast inference.

### Tips for better results
- **More data:** Use a larger dataset (e.g., v0.2 when available)
- **More epochs:** 5-10 epochs can improve performance, but watch for overfitting
- **Larger model:** 2B or 4B models will generally perform better than 0.8B
- **Lower learning rate:** If training is unstable, try `1e-4` or `5e-5`
- **Higher LoRA rank:** Try `LORA_R=64` or `LORA_R=128` for more capacity